# Charting coronaviruse
> charting out my very own questions

- toc: true
- badges: true
- comments: true
- author: Khalid Omar
- categories: [python]
- permalink: /covid19/
- hide: true

In [1]:
#hide
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import altair as alt
import requests

# in progress, go back!

## Data

Loading data from [the john hopkins data repo], specifically the [time series data](https://github.com/CSSEGISandData/COVID-19/tree/master/csse_covid_19_data/csse_covid_19_time_series).

- Deaths
- Recovered
- Confirmed

In [37]:
#collapse
def get_timeseries_data():
    """Downloads the three kinds of timeseries data
    and returns a dict with three dataframes"""

    data = {}
    data_files = ["Confirmed", "Deaths", "Recovered"]
    baseurl = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_19-covid-"

    column_map = {'Province/State': 'state', 
                  'Country/Region': 'country',
                   'Lat': 'lat', 'Long': 'long'}

    for name in data_files:
        url = f"{baseurl}{name}.csv"
        df = pd.read_csv(url)
        print(f"Downloaded: {name} with shape {df.shape}")

        df.rename(columns=column_map, inplace=True)
        df["type"] = name.lower() # type of case
        
        data[name] = df

    return data

data = get_timeseries_data()
df = data["Confirmed"].copy()
df.head()

Downloaded: Confirmed with shape (501, 66)
Downloaded: Deaths with shape (501, 66)
Downloaded: Recovered with shape (501, 66)


,state,country,lat,long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,3/15/20,3/16/20,3/17/20,3/18/20,3/19/20,3/20/20,3/21/20,3/22/20,3/23/20,type
0,NaN,Thailand,15.0000,101.0000,2,3,5,7,8,8,...,114,147,177,212,272,322,411,599,599.0,confirmed
1,NaN,Japan,36.0000,138.0000,2,1,2,2,4,4,...,839,825,878,889,924,963,1007,1086,1086.0,confirmed
2,NaN,Singapore,1.2833,103.8333,0,1,3,3,4,5,...,226,243,266,313,345,385,432,455,455.0,confirmed
3,NaN,Nepal,28.1667,84.2500,0,0,0,1,1,1,...,1,1,1,1,1,1,1,2,2.0,confirmed
4,NaN,Malaysia,2.5000,112.5000,0,0,0,3,4,4,...,428,566,673,790,900,1030,1183,1306,1306.0,confirmed


In [39]:
# convert date columns into one date col with values
def process_timeseries(df):
    col_map = {'Province/State': 'state', 'Country/Region': 'country'}
    df.rename(columns=col_map, inplace=True)
    
    name_cols = list(df.columns[:4]) + ["type"] 
    df = pd.melt(df, name_cols, var_name="Date", value_name="cases")
    
    df["date"] = pd.to_datetime(df.Date) # datetime format
    df.drop(columns="Date", inplace=True)
    
    return df

d = process_timeseries(data["Confirmed"].copy())
print(d.columns)
d.head()
#d.groupby(["country", "type", "date"]).sum()

Index(['state', 'country', 'lat', 'long', 'type', 'cases', 'date'], dtype='object')


,state,country,lat,long,type,cases,date
0,NaN,Thailand,15.0000,101.0000,confirmed,2.0,2020-01-22
1,NaN,Japan,36.0000,138.0000,confirmed,2.0,2020-01-22
2,NaN,Singapore,1.2833,103.8333,confirmed,0.0,2020-01-22
3,NaN,Nepal,28.1667,84.2500,confirmed,0.0,2020-01-22
4,NaN,Malaysia,2.5000,112.5000,confirmed,0.0,2020-01-22


In [40]:
#d.drop(columns=["lat", "long"], inplace=True)
d = d.groupby(["country", "date", "type"]).sum()
d = d.reset_index()

In [52]:
countries = ["Pakistan", "India", "Australia"]
d.query("country in @countries")

,country,date,type,lat,long,cases
496,Australia,2020-01-22,confirmed,-220.5258,1269.5003,0.0
497,Australia,2020-01-23,confirmed,-220.5258,1269.5003,0.0
498,Australia,2020-01-24,confirmed,-220.5258,1269.5003,0.0
499,Australia,2020-01-25,confirmed,-220.5258,1269.5003,0.0
500,Australia,2020-01-26,confirmed,-220.5258,1269.5003,4.0
501,Australia,2020-01-27,confirmed,-220.5258,1269.5003,5.0
502,Australia,2020-01-28,confirmed,-220.5258,1269.5003,5.0
503,Australia,2020-01-29,confirmed,-220.5258,1269.5003,6.0
504,Australia,2020-01-30,confirmed,-220.5258,1269.5003,9.0
505,Australia,2020-01-31,confirmed,-220.5258,1269.5003,9.0


In [68]:
base = alt.Chart(d.query("country in @countries")).properties()

line = base.mark_line().encode(
    x="date",
    y="cases",
    color="country"
)

line

alt.Chart(...)

## Australia Data only

From [the Guardian](https://www.theguardian.com/news/datablog/ng-interactive/2020/mar/23/how-many-cases-of-coronavirus-are-there-in-australia-live-statistics), which is collecting the data from different sources and putting it all into a Google sheet.

In [108]:
#collapse
# google sheets and json
# https://www.theguardian.com/news/datablog/ng-interactive/2020/mar/23/how-many-cases-of-coronavirus-are-there-in-australia-live-statistics

aus_json_data = "https://interactive.guim.co.uk/docsdata/1q5gdePANXci8enuiS4oHUJxcxC13d6bjMRSicakychE.json"

r = requests.get(aus_json_data)
aus = r.json()
# now we have a dict object with one key, sheets
print(aus['sheets']['about'][0]['about'])
print(aus['sheets'].keys())

aus_totals = pd.DataFrame.from_records(aus['sheets']['latest totals'])
aus_updates = pd.DataFrame.from_records(aus['sheets']["updates"])

aus_totals

This data has been compiled by Guardian Australia from official state and territory media releases and websites. Some death dates and figures are from media reports. We assign cases to the date on which they were reported by the health department, and deaths are assigned to the date they occured. Extended data on testing and demographics varies between each state and territory so may not always be available. Please contact nick.evershed@theguardian.com if you spot an error in the data or to make a suggestion. This data is released under a Attribution 3.0 Australia (CC BY 3.0 AU) license, which means it is ok to re-use but please provide attribution and a link to Guardian Australia
dict_keys(['updates', 'latest totals', 'sources', 'about', 'data validation'])
(193, 17)


,State or territory,Long name,Confirmed cases (cumulative),Deaths,Tests conducted,Tests per million,Last updated
0,NSW,New South Wales,818,7,61848,7619,2020-03-24
1,VIC,Victoria,411,,23700,3575,2020-03-24
2,QLD,Queensland,319,,32000,6255,2020-03-23
3,SA,South Australia,134,,16717,9517,2020-03-23
4,ACT,Australian Capital Territory,39,,2221,5188,2020-03-24
5,NT,Northern Territory,6,,,,2020-03-24
6,TAS,Tasmania,28,,707,1320,2020-03-23
7,WA,Western Australia,140,1,10088,3835,2020-03-23
8,National,National,1895,8,147281,5784,2020-03-24


In [129]:
#collapse
base = alt.Chart(aus_totals)

tests = base.mark_bar().encode(
    x="State or territory",
    y="Tests conducted:Q",
    color="State or territory"
).properties(
    title="Tests Conducted"
)

cases = base.mark_bar().encode(
    x="State or territory",
    y="Confirmed cases (cumulative):Q",
    color="State or territory"
).properties(
    title="Covid cases"
)

tests | cases

alt.HConcatChart(...)